# BLIP-DRNet: Pipeline Quickstart (Clean Architecture)
Notebook này minh họa quy trình nạp cấu hình, khởi tạo mô hình LPBO-DRNet và chạy đánh giá/suy luận trực tiếp từ các module trong `src/` tuân thủ chuẩn Clean Architecture & SOLID.

In [ ]:
import sys
from pathlib import Path

# Thiết lập root path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import yaml
import torch
from src.domain.entities.dr_grade import DRGrade
from src.infrastructure.deep_learning.models.lpbo_drnet.model import LPBOBoundaryConditionedNet
from src.adapters.gateways.checkpoint_gateway import CheckpointGateway
from src.adapters.presenters.metrics_presenter import MetricsPresenter, MetricsCalculator
from src.application.use_cases.evaluate_classifier_use_case import EvaluateClassifierUseCase
print("✅ Tất cả các module Clean Architecture đã được import thành công!")

In [ ]:
# Đọc cấu hình chuẩn RS-27
config_path = ROOT / "configs/training/rs_27_best.yaml"
with open(config_path, "r") as f:
    cfg = yaml.safe_load(f)

print(f"Cấu hình tải thành công: Model={cfg.get('model', {}).get('name', 'LPBO-DRNet')}")
print(f"Số lớp DR: {cfg.get('num_classes')} | Chế độ dữ liệu: {cfg.get('data_mode')}")

In [ ]:
# Khởi tạo mô hình LPBO-DRNet
device = torch.device("cuda" if torch.cuda.is_available() else ("mps" if hasattr(torch.backends, "mps") and torch.backends.mps.is_available() else "cpu"))

m_cfg = cfg.get("model", {})
model = LPBOBoundaryConditionedNet(
    num_classes=cfg.get("num_classes", 5),
    proj_dim=m_cfg.get("proj_dim", 256),
    num_super_tokens=m_cfg.get("num_super_tokens", 16),
    num_heads=m_cfg.get("num_heads", 8),
    sa_layers=m_cfg.get("sa_layers", 2),
    dropout=m_cfg.get("head_dropout", 0.1),
    drop_path_rate=m_cfg.get("drop_path_rate", 0.1),
).to(device)
model.eval()

print(f"Mô hình đã khởi tạo trên thiết bị: {device}")

In [ ]:
# Test nhanh suy luận với ảnh giả lập (dummy tensor)
dummy_input = torch.randn(2, 3, 384, 384).to(device)
with torch.no_grad():
    probs = model.predict_proba(dummy_input)
    preds = probs.argmax(dim=-1)

for i in range(len(preds)):
    grade = DRGrade.from_int(preds[i].item())
    print(f"Mẫu {i+1}: Dự đoán = Grade {grade.grade} ({grade.name_en}) - Độ tin cậy = {probs[i][grade.grade].item()*100:.2f}%")